In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

In [4]:
df  = pd.read_csv("../artifacts/raw/data.csv")

NameError: name 'pd' is not defined

In [ ]:
df.head()

In [ ]:
df["Efficiency_Status"].value_counts()

In [ ]:
df.columns

In [ ]:
df.info()

### DATA PROCESSING

In [ ]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"] , errors='coerce')

In [ ]:
categorical_cols = ['Operation_Mode','Efficiency_Status']
for col in categorical_cols:
    df[col] = df[col].astype('category')

### EDA

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(5,5))
    sns.histplot(df[col] , kde=True , bins=30)
    plt.title(f"Histogram for {col} ")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.show()

In [ ]:
sns.pairplot(df[numeric_cols])
plt.suptitle('Pair Plot for Numeric features' , y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
sns.countplot(x='Efficiency_Status' , data=df , palette='viridis')
plt.title("Efficiemmcy status countplot")
plt.show()

### FE

In [ ]:
df.head()

In [ ]:
df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Day"] = df["Timestamp"].dt.day

df["Hour"] = df["Timestamp"].dt.hour

In [ ]:
df.drop(columns=["Timestamp","Machine_ID"] , inplace=True)

In [ ]:
df.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
df["Efficiency_Target"] = label_encoder.fit_transform(df["Efficiency_Status"])

In [ ]:
label_encoder = LabelEncoder()
df["Operation_Mode"] = label_encoder.fit_transform(df["Operation_Mode"])

In [ ]:
df["Efficiency_Status"].value_counts()

In [ ]:
df["Efficiency_Target"].value_counts()

In [ ]:
df.columns

In [ ]:
features = [
    'Operation_Mode', 'Temperature_C', 'Vibration_Hz',
       'Power_Consumption_kW', 'Network_Latency_ms', 'Packet_Loss_%',
       'Quality_Control_Defect_Rate_%', 'Production_Speed_units_per_hr',
       'Predictive_Maintenance_Score', 'Error_Rate_%','Year', 'Month', 'Day', 'Hour'
]

In [ ]:
X = df[features]

In [ ]:
y = df["Efficiency_Target"]

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X_scaled, y, test_size=0.2 , random_state=42 , stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
clf = LogisticRegression(random_state=42,max_iter=1000)
clf.fit(X_train,y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score , classification_report

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
### Permutation imprtance

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
result = permutation_importance(clf , X_test , y_test , n_repeats=10 , random_state=42 , n_jobs=-1)

In [ ]:
importance_df = pd.DataFrame({
    'Feature' : features,
    'Importances' : result.importances_mean
})

In [ ]:
importance_df.sort_values(by='Importances', ascending=False)